### Imports and Configuration

In [1]:
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (path / "config" / "settings.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find config/settings.py.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import settings

keras.utils.set_random_seed(settings.RANDOM_STATE)

MODEL_PATH = settings.CLASSIFICATION_MODELS_DIR / "inceptionv3_final.keras"
CLASS_MAPPING_PATH = (
    settings.CLASSIFICATION_MODELS_DIR / "inceptionv3_class_mapping.json"
)
TEST_MANIFEST_PATH = settings.TEST_MANIFEST_PATH

REPORTS_DIR = settings.REPORTS_DIR
FIGURES_DIR = settings.FIGURES_DIR
METRICS_DIR = settings.METRICS_DIR

for directory in [REPORTS_DIR, FIGURES_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project:", settings.PROJECT_NAME)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Model path:", MODEL_PATH)
print("Test manifest:", TEST_MANIFEST_PATH)

I0000 00:00:1793092097.788469    8388 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793092097.849045    8388 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793092099.269884    8388 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Project: BrainScan_AI
TensorFlow: 2.21.0
GPU devices: []
Model path: /home/moali/projects/brainscan_ai/models/classification/inceptionv3_final.keras
Test manifest: /home/moali/projects/brainscan_ai/data/splits/test.csv


W0000 00:00:1793092100.517172    8388 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


### 2 -  Load the trained model and class mapping

In [2]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Final model not found: {MODEL_PATH}. "
        "Complete Notebook 05 first."
    )

if not CLASS_MAPPING_PATH.is_file():
    raise FileNotFoundError(
        f"Class mapping not found: {CLASS_MAPPING_PATH}."
    )

if not TEST_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Test manifest not found: {TEST_MANIFEST_PATH}. "
        "Run Notebook 02 and verify the test split."
    )

model = keras.models.load_model(MODEL_PATH, compile=False)

with open(CLASS_MAPPING_PATH, encoding="utf-8") as file:
    class_config = json.load(file)

CLASS_NAMES = [str(name) for name in class_config["classes"]]
LABEL_TO_ID = {
    str(label): int(index)
    for label, index in class_config["label_to_id"].items()
}
NUM_CLASSES = len(CLASS_NAMES)

IMAGE_HEIGHT = model.input_shape[1]
IMAGE_WIDTH = model.input_shape[2]
IMAGE_CHANNELS = model.input_shape[3]
INPUT_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS)

if model.output_shape[-1] != NUM_CLASSES:
    raise ValueError(
        f"Model outputs {model.output_shape[-1]} classes, "
        f"but the class mapping defines {NUM_CLASSES}."
    )

print("Model:", model.name)
print("Input shape:", INPUT_SHAPE)
print("Classes:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)

Model: BrainScan_InceptionV3
Input shape: (299, 299, 3)
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Number of classes: 4


In [3]:
!nvidia-smi

Tue Oct 27 09:08:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.99.02              Driver Version: 595.99.02      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 2000 Ada Gene...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   35C    P8              1W /   35W |      17MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 3 — Load and validate the test manifest

In [4]:
test_df = pd.read_csv(TEST_MANIFEST_PATH)

IMAGE_PATH_COL = "image_path"
LABEL_COL = "label"

required_columns = {IMAGE_PATH_COL, LABEL_COL}
missing_columns = required_columns - set(test_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}. "
        f"Available columns: {test_df.columns.tolist()}"
    )

if test_df.empty:
    raise ValueError("The test manifest is empty.")

if test_df[IMAGE_PATH_COL].isna().any() or test_df[LABEL_COL].isna().any():
    raise ValueError("The test manifest contains missing paths or labels.")

test_df["label_name"] = test_df[LABEL_COL].astype(str)

unknown_labels = set(test_df["label_name"]) - set(LABEL_TO_ID)

if unknown_labels:
    raise ValueError(f"Unknown test labels: {unknown_labels}")

test_df["label_id"] = test_df["label_name"].map(LABEL_TO_ID).astype("int32")


def resolve_image_path(value):
    path = Path(str(value)).expanduser()

    if not path.is_absolute():
        path = PROJECT_ROOT / path

    return path.resolve()


test_df["resolved_image_path"] = test_df[IMAGE_PATH_COL].map(
    resolve_image_path
)

missing_files = [
    str(path)
    for path in test_df["resolved_image_path"]
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} test images are missing. "
        f"Examples: {missing_files[:5]}"
    )

print("Test images:", len(test_df))
print("\nTest class distribution:")
display(
    test_df["label_name"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
    .rename("count")
    .to_frame()
)

Test images: 1054

Test class distribution:


,count
label_name,
glioma,243
meningioma,247
notumor,300
pituitary,264


### 4 — Create the test dataset

In [5]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = settings.BATCH_SIZE


def load_test_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMAGE_HEIGHT, IMAGE_WIDTH],
        antialias=True,
    )

    image = tf.cast(image, tf.float32)

    image = keras.applications.inception_v3.preprocess_input(image)

    return image, label


test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["resolved_image_path"].astype(str).to_numpy(),
        test_df["label_id"].to_numpy(dtype=np.int32),
    )
)

test_ds = test_ds.map(
    load_test_image,
    num_parallel_calls=AUTOTUNE,
)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Test dataset ready.")
print("Expected input:", model.input_shape)
print("Actual batch shape:", next(iter(test_ds))[0].shape)

Test dataset ready.
Expected input: (None, 299, 299, 3)
Actual batch shape: (32, 299, 299, 3)


### 5 — Generate predictions

In [6]:
y_true = test_df["label_id"].to_numpy()

start_time = time.perf_counter()

probabilities = model.predict(test_ds, verbose=1)

total_inference_time = time.perf_counter() - start_time

y_pred = np.argmax(probabilities, axis=1)
confidence = np.max(probabilities, axis=1)

assert len(y_true) == len(y_pred), "Prediction count mismatch."

print("Predictions generated:", len(y_pred))
print(f"Total prediction time: {total_inference_time:.3f} seconds")
print(f"Average time per image: {total_inference_time / len(y_pred):.6f} seconds")

33/33 ━━━━━━━━━━━━━━━━━━━━ 31s 908ms/step
Predictions generated: 1054
Total prediction time: 31.526 seconds
Average time per image: 0.029911 seconds


### 6 — Calculate classification metrics

### 7 — Per-class classification report

### 8 — Plot the confusion matrix

### 9 — Save predictions and inspect mistakes

### 10 — Visualize misclassified images

### 11 — Export the final evaluation summary